# 12.5 - Itertools => Accumulate, Pairwise, Batched & Tee

## `accumulate(iterable[, function, *, initial=None])`

Yields **running results**. The default function is addition.

```python
from itertools import accumulate
import operator

list(accumulate([1, 2, 3, 4, 5]))                  # [1, 3, 6, 10, 15]  running sum
list(accumulate([1, 2, 3, 4, 5], operator.mul))    # [1, 2, 6, 24, 120] running product
list(accumulate([3, 1, 4, 1, 5], max))             # [3, 3, 4, 4, 5]    running maximum
```

- Any function that takes two arguments works: `max`, `min`, `operator.mul`, or your own.
- `initial=` adds a starting value. The output then has **one more item** than the input.

```python
list(accumulate([1, 2, 3], initial=100))   # [100, 101, 103, 106]
```

- `functools.reduce()` returns only the **final** value. `accumulate` returns every step.

## `pairwise(iterable)` (Python 3.10+)

Yields **overlapping pairs** of neighbours.

```python
from itertools import pairwise

list(pairwise("ABCDE"))     # [('A', 'B'), ('B', 'C'), ('C', 'D'), ('D', 'E')]
```

- The output has one item fewer than the input.
- With fewer than two items, the output is empty.
- Typical use: differences between consecutive values.

## `batched(iterable, n, *, strict=False)` (Python 3.12+)

Splits an iterable into **tuples of length `n`**. The last tuple may be shorter.

```python
from itertools import batched

list(batched("ABCDEFG", 3))     # [('A', 'B', 'C'), ('D', 'E', 'F'), ('G',)]
```

- It is lazy: it reads only enough input to fill one batch.
- `strict=True` (Python 3.13+) raises `ValueError` if the last batch is shorter than `n`.
- `n` must be at least 1.
- On Python 3.11 and older, `itertools.batched` does not exist. A short replacement is shown in the code cell.

## `tee(iterable, n=2)`

Splits **one** iterator into `n` independent iterators.

```python
from itertools import tee

first, second = tee([1, 2, 3])
list(first)     # [1, 2, 3]
list(second)    # [1, 2, 3]
```

- After calling `tee`, **do not use the original iterator** anymore.
- `tee` stores values that one copy has read and the others have not. This can use a lot of memory.
- If one copy is fully consumed before the others start, `list()` is usually simpler and faster.
- The iterators returned by `tee` are **not thread-safe**.

## Key Rules

- `accumulate` = every step, `reduce` = final result.
- `initial=` changes the output length.
- Use `pairwise` for neighbours and `batched` for non-overlapping chunks.
- Prefer a `list` over `tee` when the data fits in memory.

## Source

https://docs.python.org/3/library/itertools.html#itertools.accumulate

https://docs.python.org/3/library/itertools.html#itertools.pairwise

https://docs.python.org/3/library/itertools.html#itertools.batched

https://docs.python.org/3/library/itertools.html#itertools.tee

In [ ]:
import itertools
import operator
import sys
from itertools import accumulate, pairwise, tee, islice

# accumulate: running results
data = [1, 2, 3, 4, 5]
print(list(accumulate(data)))                        # running sum
print(list(accumulate(data, operator.mul)))          # running product
print(list(accumulate([3, 1, 4, 1, 5], max)))        # running maximum
print(list(accumulate([3, 1, 4, 1, 5], min)))        # running minimum
print(list(accumulate([1, 2, 3], initial=100)))      # one extra item at the start

# A running balance
transactions = [100, -30, -20, 50]
print(list(accumulate(transactions, initial=0)))     # [0, 100, 70, 50, 100]

# pairwise: overlapping neighbours
print(list(pairwise("ABCDE")))
values = [1, 4, 9, 16]
print([b - a for a, b in pairwise(values)])          # differences: [3, 5, 7]
print(list(pairwise([1])))                           # fewer than two items: []

# batched exists from Python 3.12; provide a small fallback for older versions
if hasattr(itertools, "batched"):
    from itertools import batched
else:
    def batched(iterable, n):
        iterator = iter(iterable)
        while batch := tuple(islice(iterator, n)):
            yield batch

print(list(batched("ABCDEFG", 3)))                   # last batch may be shorter

# strict=True needs Python 3.13
if sys.version_info >= (3, 13):
    try:
        list(batched("ABCDEFG", 3, strict=True))
    except ValueError as error:
        print("strict:", type(error).__name__)

# tee: independent copies of one iterator
first, second = tee([1, 2, 3])
print(list(first), list(second))

# tee is useful to loop over the same stream twice (here: neighbours)
left, right = tee(iter("ABCD"))
next(right, None)
print(list(zip(left, right)))                        # same idea as pairwise